# **Pronóstico de las acciones de TESLA: Winters, Descomposición y ARIMA**

In [ ]:
#Instalamos la librería
!pip install pmdarima

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller #Determinar la estacionalidad
from statsmodels.tsa.stattools import kpss #Determinar la tendencia
from statsmodels.graphics.tsaplots import plot_acf #Determinar varias componentes
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import statsmodels.api as sm
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

## **1. Serie de tiempo**

In [ ]:
#Importar la serie
ts= pd.read_excel("TSLA_acciones.xlsx")
print(ts.shape)
ts.head()

Los datos son diarios del precio de cierre de las acciones de TESLA, solo de días hábiles (lunes a viernes), y van del 02/01/2020 al 30/12/2022 (756 registros)

In [ ]:
#Que tipo de datos tenemos-
ts.dtypes # Siempre tiene que ser flotante o entero y la fecha debe estar en formato fecha si no TRANSFORMAR DATOS

Por indicación del profesor la fecha no se convierte a índice: el modelo no toma en cuenta las fechas, cada índice es una sesión de bolsa (día hábil)

In [ ]:
#Convertimos a formato fecha #NO CORRERLO MAS DE UNA VEZ
#ts["Date"]=pd.to_datetime(ts["Date"])
#Convertimos el tiempo en índice
#ts=ts.set_index("Date").sort_index()
#ts.head()

In [ ]:
#Graficamos la serie de tiempo
plt.figure(figsize=(14, 4))
plt.xticks(rotation=45)
plt.plot(ts['Close'])
plt.xlabel("Tiempo (días hábiles)")
plt.ylabel("Valor por acción[$/acc]")
plt.title("Serie de acciones TESLA")
plt.show()

La serie tiene tendencia creciente fuerte de 2020 a finales de 2021 (pasa de ~28 a más de 400 dólares, hasta el día ~460) y después cae durante 2022 hasta ~123. Se ve mucho ruido y no se ve un patrón que se repita

## **2. Componentes de la serie**

**Autocorrelograma**

In [ ]:
#Obtener correlograma generao
plot_acf(ts['Close'], lags=len(ts)-1) #Lags= cantidad de coeficientes de autocorrelación
plt.xlabel("Lags o coeficientes")
plt.ylabel("Autocorrelacion")
plt.title("Correlograma")
plt.gca().set_ylim(-1.2,1.2)
plt.show()

In [ ]:
#Obtener correlograma detallado no puede pasar de 50 registros
plot_acf(ts['Close'], lags=50) #Lags= cantidad de coeficientes de autocorrelación
plt.xlabel("Lags o coeficientes")
plt.ylabel("Autocorrelacion")
plt.title("Correlograma")
plt.gca().set_ylim(-1.2,1.2)
plt.show()

La serie tiene tendencia porque los coeficientes de autocorrelación empiezan cerca de 1 y caen gradualmente a cero. No hay picos que se repitan cada cierto número de lags (por ejemplo cada 5), entonces no se ve estacionalidad

**ADF (Augmented Dickey-Fuller)**

H0: la serie tiene raíz unitaria (no es estacionaria). H1: la serie es estacionaria. Con p-value menor que 0.05 se rechaza H0; si es mayor, no se rechaza H0.


In [ ]:
#definimos la prueba
adf_test= adfuller(ts["Close"],autolag="AIC")
p_value=adf_test[1]
p_value

El ADF arroja un p-value mayor que 0.05 (aproximadamente 0.3427), así que no se rechaza H0. La prueba no proporciona evidencia suficiente para considerar estacionaria la serie de precios en niveles.


**KPSS**

Con `regression='ct'`, H0 plantea que la serie es estacionaria alrededor de una tendencia determinista; H1 plantea que no es estacionaria alrededor de esa tendencia. Un p-value menor que 0.05 permite rechazar H0.


In [ ]:
#Definimos la prueba KPSS
stat,p_value,_,_=kpss(ts["Close"],regression="ct",nlags="auto")
#ver los resultados
p_value

El KPSS entrega un p-value reportado de 0.01, por debajo de 0.05. Se rechaza H0 de estacionariedad alrededor de tendencia. Junto con ADF y el correlograma, esto respalda que la serie original no es estacionaria; no demuestra por sí solo una tendencia determinista.


**Periodograma** técnica para encontrar patrones repetitivos: Estacionalidad o Ciclicidad

In [ ]:
from scipy.signal import periodogram
frequencies, power =periodogram(ts["Close"].to_numpy())

#Grafica los resultados
plt.plot(frequencies, power)
plt.xlabel("Frecuencias (ciclos de observacion)")
plt.ylabel("Dencidad de potencia")
plt.title("Periodograma")
plt.show()

In [ ]:
#Determinar el valor de L
frequencies= frequencies[1:]
power=power[1:]

#encuentra la frequencia dominante
freq_dom=frequencies[np.argmax(power)]

#Calcular el periodo o L
L=1/freq_dom
L

#Sin embargo hay casos en los que este periodo no coincide con el correlograma entonces damos prioridad al correlograma

El periodo dominante del periodograma es de aproximadamente 756 observaciones, igual a la longitud total de la serie. No puede interpretarse como un ciclo estacional que se repite dentro de los datos; puede reflejar variación de muy baja frecuencia y cambios de nivel. En el correlograma no se observa una repetición clara cada cinco sesiones. Para comparar los modelos de clase se utilizará `L=5` como **periodicidad semanal experimental**, sin afirmar que haya una estacionalidad semanal comprobada.


## **3. Modelos**

Cada observación representa una sesión bursátil. Se estiman **7 observaciones futuras** usando el histórico completo hasta el 30/12/2022. Se conserva una evaluación tradicional 80/20 (pronóstico de 152 sesiones) para compararla con los ejercicios de clase. Al final se añade una prueba específica de 7 sesiones de cierre de 2022 para evaluar el horizonte solicitado.


## **Método de Winters (Suavización Exponencial Triple)**

### **Winters trend='add', seasonal='add'**

In [ ]:
#Definimos L
L = 5
#Definir el modelo
model = ExponentialSmoothing(ts['Close'], trend='add', seasonal='add', seasonal_periods=L)

#Entrenamos el modelo con la mayor cantidad de datos
fit = model.fit()

In [ ]:
#Determinar los parametros del modelo
fit.params

In [ ]:
#Obtenemos la serie ajustada
y_fit = fit.fittedvalues
print('y_fit = \n', y_fit.head())
print('y = \n', ts['Close'].head())

In [ ]:
#Obtenemos los pronosticos solicitados
n_steps = 7 #7 días
y_pred = fit.forecast(steps=n_steps)
print('y_pred = \n', y_pred)

In [ ]:
#Obtenemos la grafica
plt.figure(figsize=(14,4))
plt.plot(ts.index, ts['Close'], label='y', color='black')
plt.plot(y_fit.index, y_fit, label='y_fit',color='red')
plt.plot(y_pred.index, y_pred, label='y_pred',color='blue')
plt.legend()
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.title('Winters Method (add, add)')
plt.show()

### **Winters trend='add', seasonal='mul'**

In [ ]:
#Definimos L
L = 5
#Definir el modelo
model = ExponentialSmoothing(ts['Close'], trend='add', seasonal='mul', seasonal_periods=L)

#Entrenamos el modelo con la mayor cantidad de datos
fit = model.fit()

In [ ]:
#Determinar los parametros del modelo
fit.params

In [ ]:
#Obtenemos la serie ajustada
y_fit = fit.fittedvalues
print('y_fit = \n', y_fit.head())
print('y = \n', ts['Close'].head())

In [ ]:
#Obtenemos los pronosticos solicitados
n_steps = 7 #7 días
y_pred = fit.forecast(steps=n_steps)
print('y_pred = \n', y_pred)

In [ ]:
#Obtenemos la grafica
plt.figure(figsize=(14,4))
plt.plot(ts.index, ts['Close'], label='y', color='black')
plt.plot(y_fit.index, y_fit, label='y_fit',color='red')
plt.plot(y_pred.index, y_pred, label='y_pred',color='blue')
plt.legend()
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.title('Winters Method (add, mul)')
plt.show()

### **Winters trend='mul', seasonal='add'**

In [ ]:
#Definimos L
L = 5
#Definir el modelo
model = ExponentialSmoothing(ts['Close'], trend='mul', seasonal='add', seasonal_periods=L)

#Entrenamos el modelo con la mayor cantidad de datos
fit = model.fit()

In [ ]:
#Determinar los parametros del modelo
fit.params

In [ ]:
#Obtenemos la serie ajustada
y_fit = fit.fittedvalues
print('y_fit = \n', y_fit.head())
print('y = \n', ts['Close'].head())

In [ ]:
#Obtenemos los pronosticos solicitados
n_steps = 7 #7 días
y_pred = fit.forecast(steps=n_steps)
print('y_pred = \n', y_pred)

In [ ]:
#Obtenemos la grafica
plt.figure(figsize=(14,4))
plt.plot(ts.index, ts['Close'], label='y', color='black')
plt.plot(y_fit.index, y_fit, label='y_fit',color='red')
plt.plot(y_pred.index, y_pred, label='y_pred',color='blue')
plt.legend()
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.title('Winters Method (mul, add)')
plt.show()

### **Winters trend='mul', seasonal='mul'**

In [ ]:
#Definimos L
L = 5
#Definir el modelo
model = ExponentialSmoothing(ts['Close'], trend='mul', seasonal='mul', seasonal_periods=L)

#Entrenamos el modelo con la mayor cantidad de datos
fit = model.fit()

In [ ]:
#Determinar los parametros del modelo
fit.params

In [ ]:
#Obtenemos la serie ajustada
y_fit = fit.fittedvalues
print('y_fit = \n', y_fit.head())
print('y = \n', ts['Close'].head())

In [ ]:
#Obtenemos los pronosticos solicitados
n_steps = 7 #7 días
y_pred = fit.forecast(steps=n_steps)
print('y_pred = \n', y_pred)

In [ ]:
#Obtenemos la grafica
plt.figure(figsize=(14,4))
plt.plot(ts.index, ts['Close'], label='y', color='black')
plt.plot(y_fit.index, y_fit, label='y_fit',color='red')
plt.plot(y_pred.index, y_pred, label='y_pred',color='blue')
plt.legend()
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.title('Winters Method (mul, mul)')
plt.show()

**Parámetros de los modelos de Winters**

| Modelo | α (nivel) | β (tendencia) | γ (estacionalidad) |
|---|---|---|---|
| Winters (add, add) | 0.964 | 0.000 | 0.000 |
| Winters (add, mul) | 1.000 | 0.000 | 0.000 |
| Winters (mul, add) | 1.000 | 0.000 | 0.000 |
| Winters (mul, mul) | 0.990 | 0.001 | 0.004 |

- **α (smoothing_level):** controla qué tanto peso se le da al dato más reciente para actualizar el nivel. En los 4 modelos está prácticamente en 1, o sea que el nivel sigue casi por completo al último precio de cierre y casi no toma en cuenta la historia.
- **β (smoothing_trend):** controla qué tan rápido se actualiza la pendiente. Está en 0, entonces la tendencia se queda fija en su valor inicial y no se ajusta con los datos nuevos.
- **γ (smoothing_seasonal):** controla qué tanto se actualiza el patrón estacional. Está en 0, entonces el modelo no aprende ninguna estacionalidad, lo que confirma que la serie no es estacional.

Con α ≈ 1 y β, γ ≈ 0 el modelo prácticamente pronostica el último valor más una pendiente pequeña. Por eso los 7 pronósticos quedan alrededor de 122 a 124 dólares.

### **Medir y Evaluar el Performance de los modelos de Winters**

In [ ]:
#Separamos los datos en train y test
n_train = int(len(ts)*0.8)

train = ts['Close'].iloc[:n_train]
test = ts['Close'].iloc[n_train:]

train.shape, test.shape

**Winters trend='add', seasonal='add'**

In [ ]:
L = 5
#Definir el modelo
model_train = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=L)

#Entrenamos el modelo
fit_train = model_train.fit()

In [ ]:
#Obtenemos los ajustes del modelo
y_pred_train = fit_train.fittedvalues
y_pred_test = fit_train.forecast(steps=len(test))

In [ ]:
#obtenemos las metricas de error
mse_train = mean_squared_error(train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train, y_pred_train)*100
mpe_train = np.mean((train - y_pred_train) / train)*100

mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test) / test)*100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

**Winters trend='add', seasonal='mul'**

In [ ]:
L = 5
#Definir el modelo
model_train = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=L)

#Entrenamos el modelo
fit_train = model_train.fit()

In [ ]:
#Obtenemos los ajustes del modelo
y_pred_train = fit_train.fittedvalues
y_pred_test = fit_train.forecast(steps=len(test))

In [ ]:
#obtenemos las metricas de error
mse_train = mean_squared_error(train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train, y_pred_train)*100
mpe_train = np.mean((train - y_pred_train) / train)*100

mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test) / test)*100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

**Winters trend='mul', seasonal='add'**

In [ ]:
L = 5
#Definir el modelo
model_train = ExponentialSmoothing(train, trend='mul', seasonal='add', seasonal_periods=L)

#Entrenamos el modelo
fit_train = model_train.fit()

In [ ]:
#Obtenemos los ajustes del modelo
y_pred_train = fit_train.fittedvalues
y_pred_test = fit_train.forecast(steps=len(test))

In [ ]:
#obtenemos las metricas de error
mse_train = mean_squared_error(train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train, y_pred_train)*100
mpe_train = np.mean((train - y_pred_train) / train)*100

mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test) / test)*100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

**Winters trend='mul', seasonal='mul'**

In [ ]:
L = 5
#Definir el modelo
model_train = ExponentialSmoothing(train, trend='mul', seasonal='mul', seasonal_periods=L)

#Entrenamos el modelo
fit_train = model_train.fit()

In [ ]:
#Obtenemos los ajustes del modelo
y_pred_train = fit_train.fittedvalues
y_pred_test = fit_train.forecast(steps=len(test))

In [ ]:
#obtenemos las metricas de error
mse_train = mean_squared_error(train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train, y_pred_train)*100
mpe_train = np.mean((train - y_pred_train) / train)*100

mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test) / test)*100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

## **Descomposición Multiplicativa**

**Implementamos el modelo**

In [ ]:
# Realizar la descomposición multiplicativa
L = 5
model = sm.tsa.seasonal_decompose(ts['Close'], model='multiplicative', period=L)

In [ ]:
# Grafica la descomposición
fig = model.plot()
plt.show()

La descomposición multiplicativa muestra tendencia, componente estacional y residuo. Con `L=5` la componente estacional queda muy cercana a 1, lo que sugiere una variación semanal estimada pequeña. Esta descomposición describe los datos observados: **no demuestra una estacionalidad significativa ni permite afirmar que el ruido futuro será constante**.


In [ ]:
#Obtener la serie ajustada
y_fit = model.trend * model.seasonal
print('y_fit = \n', y_fit.iloc[int(L/2):int(L/2)+5]) #L/2 en adelante unos 5 datos
print('y = \n', ts['Close'].iloc[int(L/2):int(L/2)+5])

In [ ]:
#Verificar los vacíos
model.trend.iloc[int(L/2)]

In [ ]:
# Pronosticar futuros valores
n_forecast = 7

# Obtener los valores válidos de la tendencia
trend = model.trend.dropna()

# Obtener las posiciones numéricas originales de la tendencia
x = np.arange(len(ts))[model.trend.notna()]

# Ajustar una tendencia lineal
coef = np.polyfit(x, trend.values, 1)

# Pronosticar la tendencia futura
future_index = np.arange(len(ts), len(ts) + n_forecast)
trend_forecast = np.polyval(coef, future_index)

# Repetir el patrón estacional de los últimos L períodos
seasonal_forecast = np.tile(
    model.seasonal.iloc[-L:].to_numpy(),
    n_forecast // L + 1
)[:n_forecast]

# Generar el índice futuro según el tipo de serie
if isinstance(ts.index, pd.DatetimeIndex):
    freq = ts.index.freq or pd.infer_freq(ts.index)

    future_dates = pd.date_range(
        start=ts.index[-1],
        periods=n_forecast + 1,
        freq=freq
    )[1:]
else:
    future_dates = np.arange(len(ts), len(ts) + n_forecast)

# Combinar tendencia y estacionalidad multiplicativa
y_pred = pd.Series(
    trend_forecast * seasonal_forecast,
    index=future_dates
)

print('y_pred: \n', y_pred)

In [ ]:
# Gráfico de la serie histórica, valores ajustados y pronósticos
plt.figure(figsize=(14, 4))

plt.plot(ts.index, ts['Close'], label='y', color = 'black')
plt.plot(y_fit.index, y_fit, color='red', label='y_fit')
plt.plot(y_pred.index, y_pred, label='y_pred', color = 'blue')

# Separación entre histórico y futuro
plt.axvline(x=y_pred.index[0], color='gray', linestyle='--')

plt.legend()
plt.title('Descomposición multiplicativa')
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.show()

**Medir y evaluar el performance del modelo**

In [ ]:
# Dividir la serie en 80% entrenamiento y 20% prueba
n_train = int(len(ts) * 0.8)

train = ts['Close'].iloc[:n_train]
test = ts['Close'].iloc[n_train:]

train.shape, test.shape

In [ ]:
# Ajustar la descomposición multiplicativa solo con entrenamiento
L = 5

model_train = sm.tsa.seasonal_decompose(
    train, model='multiplicative', period=L
)

# Obtener los valores ajustados de entrenamiento
y_pred_train = model_train.trend * model_train.seasonal

In [ ]:
# Pronosticar la tendencia utilizando solo entrenamiento
# Obtener la tendencia válida
trend_train = model_train.trend.dropna()

# Obtener posiciones numéricas originales
x_train = np.arange(len(train))[model_train.trend.notna()]

# Ajustar tendencia lineal
coef_train = np.polyfit(x_train, trend_train.values, 1)

# Pronosticar la tendencia para Test
future_index = np.arange(len(train), len(train) + len(test))

trend_forecast_test = np.polyval(coef_train, future_index)

# Repetir el patrón estacional de entrenamiento
seasonal_forecast_test = np.tile(
    model_train.seasonal.iloc[-L:].to_numpy(),
    len(test) // L + 1
)[:len(test)]

# Pronosticar Test combinando tendencia y estacionalidad
y_pred_test = pd.Series(
    trend_forecast_test * seasonal_forecast_test,
    index=test.index
)

In [ ]:
# Calcular métricas de evaluación de train
# Excluir los valores sin tendencia estimada
valid_train = y_pred_train.notna()

train_valid = train.loc[valid_train]
y_pred_train_valid = y_pred_train.loc[valid_train]

mse_train = mean_squared_error(train_valid, y_pred_train_valid)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train_valid, y_pred_train_valid) * 100
mpe_train = np.mean((train_valid - y_pred_train_valid) / train_valid) * 100

# Calcular métricas de evaluación de test
mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test) * 100
mpe_test = np.mean((test - y_pred_test) / test) * 100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

## **Descomposición Aditiva**

**Implementamos el modelo**

In [ ]:
# Realizar la descomposición aditiva
L = 5
model = sm.tsa.seasonal_decompose(ts['Close'], model='additive', period=L)

In [ ]:
# Grafica la descomposición
fig = model.plot()
plt.show()

La descomposición aditiva muestra una componente estacional pequeña alrededor de cero y residuos variables. Al comparar ambas descomposiciones, la amplitud semanal estimada parece limitada. **No se debe confundir la amplitud estacional con la volatilidad total del precio**.


In [ ]:
#Obtener la serie ajustada
y_fit = model.trend + model.seasonal
print('y_fit = \n', y_fit.iloc[int(L/2):int(L/2)+5]) #L/2 en adelante unos 5 datos
print('y = \n', ts['Close'].iloc[int(L/2):int(L/2)+5])

In [ ]:
#Verificar los vacíos
model.trend.iloc[int(L/2)]

In [ ]:
# Pronosticar futuros valores
n_forecast = 7

# Obtener los valores válidos de la tendencia
trend = model.trend.dropna()

# Obtener las posiciones numéricas originales de la tendencia
x = np.arange(len(ts))[model.trend.notna()]

# Ajustar una tendencia lineal
coef = np.polyfit(x, trend.values, 1)

# Pronosticar la tendencia futura
future_index = np.arange(len(ts), len(ts) + n_forecast)
trend_forecast = np.polyval(coef, future_index)

# Repetir el patrón estacional de los últimos L períodos
seasonal_forecast = np.tile(
    model.seasonal.iloc[-L:].to_numpy(),
    n_forecast // L + 1
)[:n_forecast]

# Generar el índice futuro según el tipo de serie
if isinstance(ts.index, pd.DatetimeIndex):
    freq = ts.index.freq or pd.infer_freq(ts.index)

    future_dates = pd.date_range(
        start=ts.index[-1],
        periods=n_forecast + 1,
        freq=freq
    )[1:]
else:
    future_dates = np.arange(len(ts), len(ts) + n_forecast)

# Combinar tendencia y estacionalidad aditiva
y_pred = pd.Series(
    trend_forecast + seasonal_forecast,
    index=future_dates
)

print('y_pred: \n', y_pred)

In [ ]:
# Gráfico de la serie histórica, valores ajustados y pronósticos
plt.figure(figsize=(14, 4))

plt.plot(ts.index, ts['Close'], label='y', color = 'black')
plt.plot(y_fit.index, y_fit, color='red', label='y_fit')
plt.plot(y_pred.index, y_pred, label='y_pred', color = 'blue')

# Separación entre histórico y futuro
plt.axvline(x=y_pred.index[0], color='gray', linestyle='--')

plt.legend()
plt.title('Descomposición aditiva')
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.show()

**Medir y evaluar el performance del modelo**

In [ ]:
# Dividir la serie en 80% entrenamiento y 20% prueba
n_train = int(len(ts) * 0.8)

train = ts['Close'].iloc[:n_train]
test = ts['Close'].iloc[n_train:]

train.shape, test.shape

In [ ]:
# Ajustar la descomposición aditiva solo con entrenamiento
L = 5

model_train = sm.tsa.seasonal_decompose(
    train, model='additive', period=L
)

# Obtener los valores ajustados de entrenamiento
y_pred_train = model_train.trend + model_train.seasonal

In [ ]:
# Pronosticar la tendencia utilizando solo entrenamiento
# Obtener la tendencia válida
trend_train = model_train.trend.dropna()

# Obtener posiciones numéricas originales
x_train = np.arange(len(train))[model_train.trend.notna()]

# Ajustar tendencia lineal
coef_train = np.polyfit(x_train, trend_train.values, 1)

# Pronosticar la tendencia para Test
future_index = np.arange(len(train), len(train) + len(test))

trend_forecast_test = np.polyval(coef_train, future_index)

# Repetir el patrón estacional de entrenamiento
seasonal_forecast_test = np.tile(
    model_train.seasonal.iloc[-L:].to_numpy(),
    len(test) // L + 1
)[:len(test)]

# Pronosticar Test combinando tendencia y estacionalidad
y_pred_test = pd.Series(
    trend_forecast_test + seasonal_forecast_test,
    index=test.index
)

In [ ]:
# Calcular métricas de evaluación de train
# Excluir los valores sin tendencia estimada
valid_train = y_pred_train.notna()

train_valid = train.loc[valid_train]
y_pred_train_valid = y_pred_train.loc[valid_train]

mse_train = mean_squared_error(train_valid, y_pred_train_valid)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train_valid, y_pred_train_valid) * 100
mpe_train = np.mean((train_valid - y_pred_train_valid) / train_valid) * 100

# Calcular métricas de evaluación de test
mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test) * 100
mpe_test = np.mean((test - y_pred_test) / test) * 100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

## **Auto ARIMA**

**Implementamos el modelo**

In [ ]:
# Ajustar el modelo Autorima
L=5
model = auto_arima(ts['Close'], m=L, suppress_warnings=True, stepwise=True)
#Vemos el modelo resultante
model

Para el **ajuste final con toda la serie**, `auto_arima` seleccionó **ARIMA(0,1,0)** sin intercepto ni componentes estacionales: `seasonal_order=(0,0,0,5)`.

Ecuación: **ŷ(t) = y(t−1)** (pronóstico sin cambio) y **y(t) = y(t−1) + ε(t)** como representación del proceso, con varianza del error estimada por Python. La varianza `sigma2` no es una constante de tendencia. En la evaluación se ajusta otro modelo solo con entrenamiento; es necesario comprobar su estructura, ya que se puede seleccionar otra orden.


In [ ]:
# Mostrar el resumen del modelo
print(model.summary())

In [ ]:
# Obtener los valores ajustados
y_fit = model.predict_in_sample()

print('y_fit = \n', y_fit.head())
print('y = \n', ts['Close'].head())

In [ ]:
# Pronosticar futuros valores
n_forecast = 7

# Obtener los pronósticos
y_pred = model.predict(n_periods=n_forecast)

print('y_pred: \n', y_pred)

In [ ]:
# Gráfico de la serie histórica, valores ajustados y pronósticos
plt.figure(figsize=(14, 4))

plt.plot(ts.index, ts['Close'], label='y', color='black')
plt.plot(y_fit.index[1:], y_fit.iloc[1:], color='red', label='y_fit')
plt.plot(y_pred.index, y_pred, label='y_pred', color='blue')

# Separación entre histórico y futuro
plt.axvline(x=y_pred.index[0], color='gray', linestyle='--')

plt.legend()
plt.title('Auto ARIMA')
plt.xlabel('Tiempo (días hábiles)')
plt.ylabel('Valor por acción[$/acc]')
plt.show()

**Evaluar y validar el modelo**

In [ ]:
# Dividir la serie en 80% entrenamiento y 20% prueba
n_train = int(len(ts) * 0.8)

train = ts['Close'].iloc[:n_train]
test = ts['Close'].iloc[n_train:]

train.shape, test.shape

In [ ]:
# Ajustar Auto ARIMA solo con los datos de entrenamiento
L = 5

model_train = auto_arima(
    train,
    m=L,
    stepwise=True,
    suppress_warnings=True
)

print('Modelo ARIMA en entrenamiento:', model_train.order, model_train.seasonal_order)

# Obtener los valores ajustados de entrenamiento
y_pred_train = model_train.predict_in_sample()

# Pronosticar los valores del conjunto de prueba
y_pred_test = model_train.predict(n_periods=len(test))

# Conservar el índice original de Test
y_pred_test = pd.Series(
    np.asarray(y_pred_test),
    index=test.index
)

In [ ]:
# Excluir las primeras observaciones de inicialización
# según la diferenciación del modelo seleccionado
order = model_train.order
seasonal_order = model_train.seasonal_order

n_inicio = order[1] + seasonal_order[1] * seasonal_order[3]

# Seleccionar los valores válidos de entrenamiento
train_valid = train.iloc[n_inicio:]
y_pred_train_valid = y_pred_train.iloc[n_inicio:]

# Calcular métricas de evaluación de train
mse_train = mean_squared_error(train_valid, y_pred_train_valid)
rmse_train = np.sqrt(mse_train)
mape_train = mean_absolute_percentage_error(train_valid, y_pred_train_valid)*100
mpe_train = np.mean((train_valid - y_pred_train_valid) / train_valid)*100

# Calcular métricas de evaluación de test
mse_test = mean_squared_error(test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mape_test = mean_absolute_percentage_error(test, y_pred_test)*100
mpe_test = np.mean((test - y_pred_test) / test)*100

# Mostrar métricas de evaluación de train
print("Métricas de evaluación de train:")
print(f"MSE_train: {mse_train:.2f}")
print(f"RMSE_train: {rmse_train:.2f}")
print(f"MAPE_train: {mape_train:.2f}%")
print(f"MPE_train: {mpe_train:.2f}%")
print()

# Mostrar métricas de evaluación de test
print("Métricas de evaluación de test:")
print(f"MSE_test: {mse_test:.2f}")
print(f"RMSE_test: {rmse_test:.2f}")
print(f"MAPE_test: {mape_test:.2f}%")
print(f"MPE_test: {mpe_test:.2f}%")

## **Tabla comparativa original (20% final = 152 sesiones de prueba)**

| Modelo | MSE | RMSE | MAPE | MPE |
|---|---:|---:|---:|---:|
| Winters (add, add) | 3414.62 | 58.43 | 24.06% | -5.88% |
| Winters (add, mul) | 3402.90 | 58.33 | 24.00% | -5.58% |
| Winters (mul, add) | 3107.89 | 55.75 | 20.09% | 6.56% |
| Winters (mul, mul) | 3090.27 | 55.59 | 20.06% | 6.40% |
| Descomposición multiplicativa | 33494.74 | 183.02 | 84.09% | -84.09% |
| Descomposición aditiva | 33497.53 | 183.02 | 84.10% | -84.10% |
| ARIMA | 3082.61 | 55.52 | 20.21% | 5.82% |

**Interpretación de clase:** RMSE y MAPE menores indican mejor error; MPE más cercano a 0 indica menor sesgo. Con `MPE = promedio((real − pronóstico)/real)*100`, MPE positivo significa **subestimación** y MPE negativo significa **sobreestimación**. En el criterio usado en clase, un MPE entre -5% y +5% se considera sin sesgo relevante. Esta tabla describe **un único pronóstico de 152 pasos**, no una validación directa de 7 días.


**Evaluación original de 152 sesiones:** (a) menor MAPE: Winters (mul,mul), 20.06%; (b) menor RMSE: ARIMA, 55.52; (c) menor sesgo absoluto: Winters (add,mul), MPE -5.58%; (d) mayor sesgo absoluto: descomposición aditiva, MPE -84.10% (sobreestima). **Ninguno** cae dentro del intervalo -5% a +5%.

Para tomar una decisión sobre el objetivo real de TESLA también se debe leer la evaluación final de **7 sesiones**.


**8. Elección según horizonte:** En la comparación original de 152 pasos, ARIMA obtuvo el menor RMSE (55.52), aunque Winters (mul,mul) tuvo un MAPE ligeramente menor (20.06%). **No es suficiente para afirmar que ARIMA será el más preciso a 7 pasos.** Se agrega una validación de 7 sesiones al final para seleccionar el ganador de ese horizonte.


**9. Decisión de inversión:** El ARIMA final sobre la muestra completa pronostica 123.18 USD constantes para las siete sesiones siguientes al 30/12/2022. Esto describe una **ausencia de cambio esperado según el modelo**, no una garantía de precio ni una señal automática de compra, venta o mantenimiento. Los errores de prueba y la posibilidad de variaciones de mercado impiden fundamentar una recomendación firme de inversión solamente con este pronóstico. Conviene contrastar los escenarios con riesgo, plazo y más información sobre la empresa.


**10. Amplitud de las descomposiciones:** Con `L=5`, tanto la componente aditiva como los factores estacionales multiplicativos estimados son pequeños, y su patrón semanal no muestra un crecimiento claro en amplitud. Esto habla de la **amplitud de la componente estacional estimada**, no de toda la amplitud o volatilidad futura del precio. Las extrapolaciones lineales de las descomposiciones (~328 USD) son muy superiores al último cierre de 123.18 USD y reflejan una mala adaptación de la tendencia global al giro observado en 2022.


## **Validación adicional: los siguientes 7 días de prueba**

Esta sección mantiene el estilo de la práctica de clase. Usamos **los primeros 749 registros como train** y reservamos **los últimos 7 como test**, sin mezclarlos. Así cada modelo pronostica exactamente 7 sesiones no vistas (21 al 30 de diciembre de 2022). Usamos el mismo `L=5` experimental para Winters y descomposición. Después comparamos RMSE, MAPE y MPE con el criterio usado en clase. El pronóstico futuro de enero de 2023 sigue siendo el calculado arriba con los 756 registros.


In [ ]:
# Validación de los últimos 7 registros (sin modificar los modelos anteriores)
train_7 = ts['Close'].iloc[:-7]
test_7 = ts['Close'].iloc[-7:]
L = 5
resultados_7 = []
pronosticos_7 = pd.DataFrame(index=test_7.index)

# Winters: las cuatro combinaciones que vimos en clase
for tendencia in ['add', 'mul']:
    for estacionalidad in ['add', 'mul']:
        nombre = 'Winters ' + tendencia + '/' + estacionalidad
        modelo_7 = ExponentialSmoothing(train_7, trend=tendencia,
                                         seasonal=estacionalidad,
                                         seasonal_periods=L)
        ajuste_7 = modelo_7.fit()
        pred_7 = pd.Series(np.asarray(ajuste_7.forecast(steps=7)), index=test_7.index)
        pronosticos_7[nombre] = pred_7
        mse = mean_squared_error(test_7, pred_7)
        resultados_7.append([nombre, mse, np.sqrt(mse),
                             mean_absolute_percentage_error(test_7, pred_7)*100,
                             np.mean((test_7-pred_7)/test_7)*100])

# Descomposición aditiva y multiplicativa (mismo procedimiento que en clase)
for tipo in ['additive', 'multiplicative']:
    nombre = 'Descomposición ' + tipo
    descomp_7 = sm.tsa.seasonal_decompose(train_7, model=tipo, period=L)
    tendencia_7 = descomp_7.trend.dropna()
    x_7 = np.arange(len(train_7))[descomp_7.trend.notna()]
    coef_7 = np.polyfit(x_7, tendencia_7.values, 1)
    tendencia_futura_7 = np.polyval(coef_7, np.arange(len(train_7), len(train_7)+7))
    estacionalidad_futura_7 = np.tile(descomp_7.seasonal.iloc[-L:].to_numpy(), 2)[:7]
    valores_7 = (tendencia_futura_7 + estacionalidad_futura_7 if tipo == 'additive'
                 else tendencia_futura_7 * estacionalidad_futura_7)
    pred_7 = pd.Series(valores_7, index=test_7.index)
    pronosticos_7[nombre] = pred_7
    mse = mean_squared_error(test_7, pred_7)
    resultados_7.append([nombre, mse, np.sqrt(mse),
                         mean_absolute_percentage_error(test_7, pred_7)*100,
                         np.mean((test_7-pred_7)/test_7)*100])

# ARIMA: ajustar solo con train
modelo_arima_7 = auto_arima(train_7, seasonal=False, stepwise=True, suppress_warnings=True)
print('ARIMA seleccionado con train:', modelo_arima_7.order)
pred_7 = pd.Series(np.asarray(modelo_arima_7.predict(n_periods=7)), index=test_7.index)
pronosticos_7['ARIMA'] = pred_7
mse = mean_squared_error(test_7, pred_7)
resultados_7.append(['ARIMA', mse, np.sqrt(mse),
                     mean_absolute_percentage_error(test_7, pred_7)*100,
                     np.mean((test_7-pred_7)/test_7)*100])

comparacion_7 = pd.DataFrame(resultados_7, columns=['Modelo', 'MSE', 'RMSE', 'MAPE', 'MPE'])
print(comparacion_7.round(2).to_string(index=False))
print('Menor RMSE:', comparacion_7.loc[comparacion_7['RMSE'].idxmin(), 'Modelo'])
print('Menor MAPE:', comparacion_7.loc[comparacion_7['MAPE'].idxmin(), 'Modelo'])
print('MPE más cercano a cero:', comparacion_7.loc[comparacion_7['MPE'].abs().idxmin(), 'Modelo'])


In [ ]:
# Serie real y pronósticos de las últimas 7 sesiones
plt.figure(figsize=(13, 4))
plt.plot(train_7.index[-35:], train_7.iloc[-35:], label='Train', color='black')
plt.plot(test_7.index, test_7, label='Test real', color='green', marker='o')
plt.plot(test_7.index, pronosticos_7['Winters mul/add'], label='Winters mul/add', color='blue')
plt.plot(test_7.index, pronosticos_7['ARIMA'], label='ARIMA', color='red')
plt.xlabel('Tiempo (sesiones bursátiles)')
plt.ylabel('Precio de cierre (USD)')
plt.title('Validación de 7 sesiones: real vs. pronóstico')
plt.legend()
plt.show()


### **Interpretación de la evaluación de 7 días**

- **Menor RMSE y MAPE:** Winters **`trend='mul', seasonal='add'`**: RMSE **17.61 USD**, MAPE **13.32%**. De acuerdo con el criterio de MAPE usado en clase, se clasifica como **bueno (entre 10% y 20%)** en esta única ventana, aunque el error sigue siendo considerable para operar acciones.
- **Menor sesgo absoluto:** el mismo Winters (mul/add), MPE **−13.32%**; **ningún modelo** queda entre −5% y +5%. Todos los MPE son negativos y, por lo tanto, los modelos **sobreestimaron** los cierres de estas siete sesiones.
- **Mayor sesgo absoluto:** descomposición aditiva, MPE **−175.06%**; la descomposición multiplicativa también presenta fuerte sobreestimación (**−175.01%**).
- **Mejor modelo para el horizonte de siete sesiones, de acuerdo con esta comparación puntual:** Winters (mul/add), porque tiene menor RMSE y menor MAPE. No se puede asegurar que sea el mejor de forma general con una sola ventana de prueba. El mejor para 152 sesiones según RMSE fue ARIMA; el horizonte cambia el resultado.
- **Advertencia:** la evaluación conserva una sola partición cronológica de los últimos siete días, como en las prácticas de clase. Para asegurar estabilidad se necesitarían otras ventanas de validación histórica.
